In [ ]:
# --- make the safety_probes package importable ---
# Preferred: run `pip install -e .` from the repo root once; then this cell is a no-op.
# Fallback (below) handles a not-yet-installed checkout, including VSCode kernels whose working
# directory is the workspace root rather than this notebook's folder.
import sys, pathlib
try:
    import safety_probes  # already installed (editable) -> nothing to do
except ModuleNotFoundError:
    starts = []
    _f = globals().get("__vsc_ipynb_file__")          # VSCode injects the notebook's own path
    if _f:
        starts.append(pathlib.Path(_f).resolve())
    starts.append(pathlib.Path.cwd())                 # classic Jupyter: cwd = notebook dir
    for _s in starts:
        _root = next((p for p in [_s, *_s.parents]
                      if (p / "safety_probes" / "__init__.py").exists()), None)
        if _root:
            sys.path.insert(0, str(_root)); break
    import safety_probes
print("safety_probes:", safety_probes.__file__)

# Cross-behaviour summary — §4 verdict

Run **after** `deception.ipynb` and `high_stakes.ipynb` (they persist their detection + steering
numbers to `exported_probes/`). Reproduces the headline read/write dissociation test:
does **LR read best** ([Apollo App. D.1]) while **MM writes best** ([GoT Table 2])?

In [ ]:
import pandas as pd
from safety_probes import runtime, summary
runtime.PROJECT_ROOT  # exported_probes/ resolves under here

In [ ]:
# Cross-behaviour comparison: who READS best (detection AUROC) vs who WRITES best (steering NIE).
# DETECTION — AUROC by probe (higher = better reader)
detect = pd.concat([summary.auroc_pivot(summary.load_detection("deception"), "deception"),
                    summary.auroc_pivot(summary.load_detection("high-stakes"), "high-stakes")], axis=1)
print("=== DETECTION - AUROC by probe (higher = better reader) ===")
display(detect.round(3))
best_readers = detect.idxmax()
print("Best probe per eval set:"); print(best_readers.to_string(), "\n")

In [ ]:
# STEERING - NIE at coef +1 by direction (higher = better writer)
steer = pd.concat([summary.load_steering("deception"), summary.load_steering("high-stakes")], axis=1)
print("=== STEERING - NIE at coef +1 by direction (higher = better writer) ===")
display(steer.round(3))
best_writers = steer.idxmax()
print("Best direction per eval set:"); print(best_writers.to_string(), "\n")

In [ ]:
# The headline verdict [Apollo App. D.1] x [GoT Table 2]: does LR read best while MM writes best?
lr_reads = best_readers.str.startswith("LR").mean()
mm_writes = (best_writers == "MM").mean()
print(f"LR-family wins {lr_reads:.0%} of detection columns; MM wins {mm_writes:.0%} of steering columns.")
print("Read/write dissociation " + ("HOLDS" if lr_reads > 0.5 and mm_writes > 0.5 else "does NOT clearly hold")
      + " on these runs (see the dose-response plots and judge coherence before trusting the steering side).")